    # SAM3 Parameter Sweep

    SAM 3 behaves more like prompt-conditioned segmentation than a thresholded detector.
    In this setup, the practical sweep axis is **which text prompt** you give it.

    `None` means "let the VLM generate the prompt first" using the model configured in `params.py`.
    Strings and lists are passed straight through to `sam3_mask(...)`.
    


In [ ]:
%matplotlib inline

import gc
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in (cwd, *cwd.parents):
    if (candidate / 'data').exists() and (candidate / 'change_detection_results').exists():
        PROJECT_ROOT = candidate.resolve()
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate the repo root from cwd={cwd}. "
        "Expected a parent directory containing both 'data' and 'change_detection_results'."
    )

CHANGE_DIR = PROJECT_ROOT / 'change_detection_results'
SWEEP_DIR = CHANGE_DIR / 'param_sweeps'
for path in (CHANGE_DIR, SWEEP_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device.upper()}')

In [ ]:
from params import SAM3_BASELINE
from sweep_utils import load_dataset_pair, prompt_label, show_image_pair, show_sweep
from test_change_detection import sam3_mask

DATASET = 'new3'   # <- change dataset here

project_root, data_dir, original_img, edited_img, original_path, edited_path = load_dataset_pair(DATASET)
checkpoint_path = project_root / SAM3_BASELINE['checkpoint']
assert checkpoint_path.exists(), f'SAM3 checkpoint not found: {checkpoint_path}'

baseline_prompt = SAM3_BASELINE['text_prompt']
baseline_parts = [part.strip() for part in str(baseline_prompt).split(',') if part.strip()] if baseline_prompt is not None else []

print(f'Dataset: {DATASET}')
print(f'Checkpoint: {checkpoint_path}')
print(SAM3_BASELINE)
print(f'Baseline prompt parts: {baseline_parts}')


In [ ]:
show_image_pair(original_img, edited_img, title=f'{DATASET}: original vs edited')


    ## Sweep 1 - prompt source comparison

    This compares VLM-generated prompting against the baseline prompt string and a few
    variants derived from it automatically.
    


In [ ]:
prompt_candidates = [None]
if baseline_prompt is not None:
    prompt_candidates.append(baseline_prompt)
if len(baseline_parts) == 1:
    prompt_candidates.append(baseline_parts[0])
elif len(baseline_parts) > 1:
    prompt_candidates.extend(baseline_parts)
    prompt_candidates.append(baseline_parts)
    prompt_candidates.append(', '.join(reversed(baseline_parts)))

# preserve order while dropping duplicates
deduped = []
seen = set()
for prompt in prompt_candidates:
    key = repr(prompt)
    if key not in seen:
        seen.add(key)
        deduped.append(prompt)

prompt_results = {}
for prompt in deduped:
    mask, diff_map = sam3_mask(
        original_img, edited_img,
        checkpoint=str(checkpoint_path),
        text_prompt=prompt,
        vlm_model=SAM3_BASELINE['vlm_model'],
    )
    label = prompt_label(prompt)
    prompt_results[label] = {
        'mask': mask,
        'diff_map': diff_map,
        'subtitle': 'prompt source',
    }

show_sweep(
    prompt_results,
    edited_img,
    title=f'SAM3 prompt-source comparison - {DATASET}',
    baseline_key=prompt_label(SAM3_BASELINE['text_prompt']),
)


    ## Sweep 2 - custom prompt bank

    Edit `CUSTOM_PROMPTS` with nouns or short phrases that match the type of edit in your dataset.
    


In [ ]:
CUSTOM_PROMPTS = [
    SAM3_BASELINE['text_prompt'],
    'changed object',
    'edited object',
    'new object, removed object',
]

custom_results = {}
for prompt in CUSTOM_PROMPTS:
    mask, diff_map = sam3_mask(
        original_img, edited_img,
        checkpoint=str(checkpoint_path),
        text_prompt=prompt,
        vlm_model=SAM3_BASELINE['vlm_model'],
    )
    label = prompt_label(prompt)
    custom_results[label] = {
        'mask': mask,
        'diff_map': diff_map,
        'subtitle': 'manual prompt',
    }

show_sweep(
    custom_results,
    edited_img,
    title=f'SAM3 custom prompt sweep - {DATASET}',
    baseline_key=prompt_label(SAM3_BASELINE['text_prompt']),
)


    ## Combos

    Keep your best prompt candidates here for quick re-comparison after you tweak the custom bank.
    


In [ ]:
best_prompts = {
    'baseline': SAM3_BASELINE['text_prompt'],
    'auto-vlm': None,
}

if baseline_parts:
    best_prompts['parts-list'] = baseline_parts if len(baseline_parts) > 1 else baseline_parts[0]

combo_results = {}
for name, prompt in best_prompts.items():
    mask, diff_map = sam3_mask(
        original_img, edited_img,
        checkpoint=str(checkpoint_path),
        text_prompt=prompt,
        vlm_model=SAM3_BASELINE['vlm_model'],
    )
    combo_results[name] = {
        'mask': mask,
        'diff_map': diff_map,
        'subtitle': prompt_label(prompt),
    }

show_sweep(combo_results, edited_img, title=f'SAM3 combo comparison - {DATASET}', baseline_key='baseline')


In [ ]:
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()
